# EXP004 · Antisana + Zhurucay with several foundation models (Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbnarvae/LandClassification_BRIDGE/blob/colab-exp004/colab/exp004_multi_fm_smoke.ipynb)

Repeats the EXP003 smoke test (HLS 2023 dry + wet composites → **frozen encoder** → 480 m patch embeddings →
PCA 50 → k-means → scored against MAATE ecosystems and land cover 2022) for several encoders instead of
Prithvi alone:

| model | params | input handling |
|---|---|---|
| Prithvi-EO-2.0 300M-TL | 304 M | both seasons together (native multi-temporal) |
| Prithvi-EO-2.0 600M-TL | 631 M | same; 14-px patches pooled to the 16-px grid |
| TerraMind v1 small / base | 22 / 86 M | each season separately, averaged; TerraMind S2-L2A normalisation |
| DOFA base / large | 111 / 337 M | each season separately, averaged; wavelength-conditioned |
| *optional:* Satlas Swin-B Landsat, SSL4EO-L ResNet-50 | 88 / 26 M | reference encoders, pooled to the 16-px grid |

**Before running** (once): on your Mac run `bash colab/pack_inputs.sh` and upload
`data/colab/exp003_inputs.zip` to Google Drive → `MyDrive/BRIDGE_colab/`.

**Runtime:** *Runtime → Change runtime type → T4 GPU* (L4 / A100 are faster but cost more compute units).
Embedding all six models takes a few minutes of GPU time; most of the session is installs and weight downloads.
Every step skips what is already on Drive, so if the session drops just run all cells again.

In [ ]:
# Settings
BRANCH = "colab-exp004"              # git branch of mbnarvae/LandClassification_BRIDGE to run
DRIVE_DIR = "/content/drive/MyDrive/BRIDGE_colab"
MODELS = "prithvi_eo2_300_tl,prithvi_eo2_600_tl,terramind_v1_small,terramind_v1_base,dofa_base,dofa_large"
# MODELS = "all"                     # also the optional reference encoders (Satlas, SSL4EO-L)
BATCH_SIZE = 8                       # tiles per forward pass; lower to 4 if CUDA runs out of memory
AMP = False                          # True = float16 autocast (faster, results differ slightly from EXP003)
CACHE_WEIGHTS_ON_DRIVE = False       # True keeps the ~6 GB of pretrained weights on Drive between sessions

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime → Change runtime type → GPU"

In [ ]:
import os, zipfile, pathlib
from google.colab import drive
drive.mount("/content/drive")

drive_dir = pathlib.Path(DRIVE_DIR)
OUT = drive_dir / "exp004"                      # results stay on Drive
INPUTS = pathlib.Path("/content/exp003_inputs")  # local copy: fast reads
OUT.mkdir(parents=True, exist_ok=True)
zip_path = drive_dir / "exp003_inputs.zip"
if not (INPUTS / "scores.csv").exists():
    assert zip_path.exists(), f"Upload exp003_inputs.zip to {drive_dir} first (bash colab/pack_inputs.sh on your Mac)"
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(INPUTS)
for p in sorted(INPUTS.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(INPUTS)}  {p.stat().st_size / 1e6:.0f} MB")

if CACHE_WEIGHTS_ON_DRIVE:
    os.environ["HF_HOME"] = str(drive_dir / "cache/huggingface")
    os.environ["TORCH_HOME"] = str(drive_dir / "cache/torch")

In [ ]:
# Code + dependencies (terratorch pinned to the version EXP002/EXP003 were verified with)
REPO = "/content/LandClassification_BRIDGE"
if not os.path.exists(REPO):
    !git clone -q --branch {BRANCH} https://github.com/mbnarvae/LandClassification_BRIDGE.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log -1 --oneline
!pip install -q -e . "terratorch==1.2.11" rasterio geopandas pyarrow 2>&1 | tail -n 3

In [ ]:
# Quick check: every encoder builds and returns the 14×14 patch grid (random weights, no download).
# If Colab asked to restart the session after the install, restart, then run all cells again.
!python -W ignore -m bridge_lc.models.embeddings {MODELS}

## 1 · Embeddings
One model at a time; each writes `exp004/<site>/emb_<model>.npz` on Drive. The first call per model
downloads its pretrained weights (Prithvi 600M ≈ 2.5 GB).

In [ ]:
amp = "--amp" if AMP else ""
!python -W ignore scripts/exp004_multi_fm_smoke.py embed --models {MODELS} --inputs {INPUTS} --out {OUT} --batch-size {BATCH_SIZE} {amp}

## 2 · Clustering and scores
Same patches, PCA, k values and scoring as EXP003. Pooled over both sites; per-site rows are in `scores.csv`.

In [ ]:
!python -W ignore scripts/exp004_multi_fm_smoke.py analyze --inputs {INPUTS} --out {OUT}

In [ ]:
# Sanity check: Prithvi 300M here must reproduce EXP003 (same patches, same protocol)
import pandas as pd
e3 = pd.read_csv(INPUTS / "scores.csv")
e4 = pd.read_csv(OUT / "scores.csv")
e3 = e3[e3.site == "all"].assign(features=e3.features.str.replace("prithvi_l", "prithvi_eo2_300_tl_l"))
cmp = e3.merge(e4[e4.site == "all"], on=["features", "k", "reference"], suffixes=("_exp003", "_exp004"))
cols = ["features", "k", "reference", "n_recognised_exp003", "n_recognised_exp004", "nmi_exp003", "nmi_exp004"]
print(cmp[cols].round(3).to_string(index=False))
print("max |ΔNMI| =", (cmp.nmi_exp003 - cmp.nmi_exp004).abs().max().round(4))

In [ ]:
summary = pd.read_csv(OUT / "summary.csv")
for ref, s in summary.groupby("reference"):
    print(f"\n=== {ref}: best over k (both sites)")
    print(s.drop(columns="reference").to_string(index=False))

## 3 · Figures
`model_comparison.png` (all models vs the spectral baseline), `f1_by_group.png` (per ecosystem / land-cover
group, last layer, k = 16) and per-site maps.

In [ ]:
!python -W ignore scripts/exp004_multi_fm_smoke.py figures --inputs {INPUTS} --out {OUT}
from IPython.display import Image, display
for p in sorted((OUT / "figures").glob("*.png")):
    print(p.name)
    display(Image(str(p)))

## Outputs (on Drive, `MyDrive/BRIDGE_colab/exp004/`)
- `scores.csv` — every model × layer × k × reference × site (n recognised, NMI, ARI, accuracy)
- `summary.csv` — best k per model/layer and reference
- `best_per_class.json` — per-class F1 at the best k
- `patches.parquet` — patch table with every cluster assignment
- `embed_timing.csv` — GPU seconds per model and site
- `<site>/emb_<model>.npz` — embeddings (reusable for later probes / fine-tune comparisons)

Copy `exp004/` back to `data/processed/exp004/` on your Mac to keep working locally.